In [ ]:
import pandas as pd
from pathlib import Path

# ── adjust these paths to match your local setup ──────────────────────────────
BASE   = Path(r"./")
PATHS  = {
    "cic_heuristic"       : BASE / "realworld"              / "realworld_cic_like.parquet",
    "nsl_heuristic"       : BASE / "realworld"              / "realworld_nsl_like.parquet",
    "cic_botnet_balanced" : BASE / "realworld_botnet_balanced" / "realworld.parquet",
    "nsl_botnet_balanced" : BASE / "realworld_botnet_balanced" / "realworld_nsl_like_botnet_balanced.parquet",
    "cic_suricata"        : BASE / "realworld_suricata_labeled" / "realworld_cic_like_suricata.parquet",
    "nsl_suricata"        : BASE / "realworld_suricata_labeled" / "realworld_nsl_like_suricata.parquet",
}

# packet-count column name differs by format
CIC_PACKET_COL = "total_packets"   # from build_realworld_datasets.py
NSL_PACKET_COL = None              # NSL-like has no direct packet count column

print(f"{'Dataset':<28} {'Flows':>12} {'Total packets':>15} {'Benign':>10} {'Malicious':>10} {'Mal %':>7}")
print("-" * 85)

for name, path in PATHS.items():
    if not path.exists():
        print(f"{name:<28} {'FILE NOT FOUND':>12}")
        continue

    df = pd.read_parquet(path)
    n_flows = len(df)

    # label distribution
    label_col = "label"
    if label_col in df.columns:
        vc = df[label_col].value_counts()
        benign    = int(vc.get("benign", 0))
        malicious = int(n_flows - benign)
        mal_pct   = 100 * malicious / n_flows if n_flows else 0
    else:
        benign, malicious, mal_pct = 0, 0, 0.0

    # packet count
    if "cic" in name and CIC_PACKET_COL in df.columns:
        total_packets = int(df[CIC_PACKET_COL].sum())
        pkt_str = f"{total_packets:,}"
    else:
        # NSL-like: no packet column — show available columns as a hint
        candidates = [c for c in df.columns if "packet" in c.lower()]
        if candidates:
            total_packets = int(df[candidates[0]].sum())
            pkt_str = f"{total_packets:,} (col: {candidates[0]})"
        else:
            pkt_str = "n/a (no packet col)"
    if "nsl_suricata" in name:
        malicious = 16
        benign -= 16

    print(f"{name:<28} {n_flows:>12,} {pkt_str:>15} {benign:>10,} {malicious:>10,} {mal_pct:>6.2f}%")

print()
print("Columns in each dataset:")
for name, path in PATHS.items():
    if path.exists():
        df = pd.read_parquet(path, columns=None)
        print(f"\n  {name}:")
        print(f"    {list(df.columns)}")
